# Distribución del train completo (~2M imágenes) por cluster K=70

Cuántos datos hay en cada celda de Voronoi esférica K=70 (prototipos de
`preprocessing/assets/prototipos_train-completo_k70.csv`) usando **todo el train** de `DATA_URL`.

1. Lee solo `image_id`, `panoid`, `country_code`, `latitude` y `longitude` de los `train-*.parquet` (sin imágenes).
2. Excluye la validación global (`validation_ids.parquet`), igual que `clusters_train_completo.ipynb`.
3. Asigna cada fila a su prototipo más cercano (mayor producto punto sobre la esfera, igual que los clasificadores).
4. Cuenta **imágenes** y **panoramas** por cluster, con tablas, gráficos, mapa y países dominantes.
5. Mide el desbalance y qué implica para un muestreo balanceado por celda.

In [32]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pyarrow.parquet as pq
from IPython.display import display
from plotly.subplots import make_subplots

PATRON_ARCHIVOS = "**/train-*.parquet"
PROTOTIPOS = "src/notebooks/preprocessing/assets/prototipos_train-completo_k70.csv"
VALIDACION_IDS = "datasets/samples/data/validation/validation_ids.parquet"
VALIDACION_CONFIG = "datasets/samples/data/validation/validation_config.json"
R_TIERRA = 6371.0

pd.set_option("display.max_rows", 100)

## Localizar el dataset

`DATA_URL` se toma del entorno o del `.env`; las rutas relativas se resuelven desde la raíz del proyecto.

In [33]:
env_path = next(
    (carpeta / ".env" for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el .env del proyecto desde el directorio actual.")

valor_env = ""
for linea in env_path.read_text(encoding="utf-8-sig").splitlines():
    clave, separador, valor = linea.partition("=")
    if separador and clave.strip() == "DATA_URL":
        valor_env = valor.strip().strip("\"'")

ruta_configurada = os.environ.get("DATA_URL", valor_env).strip()
if not ruta_configurada:
    raise ValueError("Configurá DATA_URL en el .env con la carpeta del dataset local.")
raiz_proyecto = env_path.parent


def resolver(ruta):
    ruta = Path(ruta).expanduser()
    return (ruta if ruta.is_absolute() else raiz_proyecto / ruta).resolve()


ruta_datos = resolver(ruta_configurada)
archivos = sorted(p for p in ruta_datos.glob(PATRON_ARCHIVOS) if p.is_file())
if not archivos:
    raise FileNotFoundError(f"No se encontraron Parquet con {PATRON_ARCHIVOS!r} en {ruta_datos}")
print(f"Dataset: {ruta_datos}")
print(f"Archivos de train: {len(archivos):,}")

Dataset: C:\Users\jurad\Documents\facultad\rrnn2\tp-final\world-streetview-500k\data
Archivos de train: 393


## 1. Metadata del train completo

Se leen solo columnas livianas. Se descartan filas de la validación global (por `image_id` o `panoid`) y coordenadas inválidas.

In [34]:
validacion = pq.read_table(resolver(VALIDACION_IDS), columns=["image_id", "panoid"]).to_pandas()
for columna in ("image_id", "panoid"):
    validacion[columna] = validacion[columna].astype("string").str.strip()
ids_ordenados = validacion.sort_values(["image_id", "panoid"]).to_numpy().tolist()
huella = hashlib.sha256(json.dumps(ids_ordenados, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()
huella_config = json.loads(resolver(VALIDACION_CONFIG).read_text(encoding="utf-8")).get("sha256_ids")
assert huella_config is None or huella_config == huella, "Los IDs de validación no coinciden con validation_config.json"
ids_val, panos_val = set(validacion["image_id"]), set(validacion["panoid"])

columnas = ["image_id", "panoid", "country_code", "latitude", "longitude"]
filas = pd.concat([pq.read_table(a, columns=columnas).to_pandas() for a in archivos], ignore_index=True)
filas = filas.rename(columns={"latitude": "lat", "longitude": "lon"})
for columna in ("image_id", "panoid"):
    filas[columna] = filas[columna].astype("string").str.strip()

validas = (np.isfinite(filas.lat) & np.isfinite(filas.lon) & filas.lat.abs().le(90) & filas.lon.abs().le(180)).to_numpy()
reservadas = (filas.image_id.isin(ids_val) | filas.panoid.isin(panos_val)).to_numpy()
n_leidas = len(filas)
train = filas.loc[validas & ~reservadas].reset_index(drop=True)
del filas

print(f"Filas leídas:            {n_leidas:,}")
print(f"Excluidas (validación):  {int(reservadas.sum()):,}")
print(f"Excluidas (coord. inv.): {int((~validas).sum()):,}")
print(f"Imágenes de train:       {len(train):,}")
print(f"Panoramas de train:      {train.panoid.nunique():,}")
print(f"Imágenes por panorama:   {train.groupby('panoid').size().value_counts().sort_index().to_dict()}")

Filas leídas:            1,999,060
Excluidas (validación):  5,332
Excluidas (coord. inv.): 0
Imágenes de train:       1,993,728
Panoramas de train:      498,432
Imágenes por panorama:   {4: 498432}


## 2. Asignación a los 70 prototipos

Misma regla que los clasificadores por celda: vector unitario en $\mathbb{R}^3$ y prototipo de mayor producto punto.

In [35]:
def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])


prototipos = pd.read_csv(resolver(PROTOTIPOS))
assert (prototipos.ind_cluster.to_numpy() == np.arange(len(prototipos))).all()
K = len(prototipos)
C = to_xyz(prototipos.lat.to_numpy(float), prototipos.lon.to_numpy(float))

X = to_xyz(train.lat.to_numpy(float), train.lon.to_numpy(float))
celda = np.empty(len(X), dtype=np.int16)
sim = np.empty(len(X))
for i in range(0, len(X), 250_000):
    s = X[i:i + 250_000] @ C.T
    celda[i:i + 250_000] = s.argmax(axis=1)
    sim[i:i + 250_000] = s.max(axis=1)
train["celda"] = celda
train["dist_km"] = np.arccos(np.clip(sim, -1, 1)) * R_TIERRA
del X, sim
print(f"K = {K}; celdas con datos: {train.celda.nunique()} de {K}")

K = 70; celdas con datos: 70 de 70


## 3. Conteo por cluster

In [36]:
g = train.groupby("celda")
top3 = (train.groupby(["celda", "country_code"]).size().rename("n").reset_index()
        .sort_values(["celda", "n"], ascending=[True, False]).groupby("celda").head(3))
top3["pct"] = top3.n / top3.celda.map(g.size())
top_paises = top3.groupby("celda").apply(
    lambda d: ", ".join(f"{c} {p:.0%}" for c, p in zip(d.country_code, d.pct)), include_groups=False)

por_celda = pd.DataFrame({
    "lat": prototipos.lat.round(2), "lon": prototipos.lon.round(2),
    "imagenes": g.size(),
    "panoramas": g["panoid"].nunique(),
    "paises": g["country_code"].nunique(),
    "dist_media_km": g["dist_km"].mean().round(0),
    "top_paises": top_paises,
}).reindex(range(K)).fillna({"imagenes": 0, "panoramas": 0, "paises": 0}).astype({"imagenes": int, "panoramas": int, "paises": int})
por_celda.index.name = "ind_cluster"
por_celda.insert(4, "pct_imagenes", (100 * por_celda.imagenes / por_celda.imagenes.sum()).round(2))

ordenado = por_celda.sort_values("imagenes", ascending=False)
ordenado.insert(0, "rank", range(1, K + 1))
ordenado.insert(6, "pct_acumulado", ordenado.pct_imagenes.cumsum().round(1))
display(ordenado)

,rank,lat,lon,imagenes,panoramas,pct_imagenes,pct_acumulado,paises,dist_media_km,top_paises
ind_cluster,,,,,,,,,,
65,1,46.17,1.93,79272,19818,3.98,4.0,6,276.0,"FR 93%, ES 3%, IT 2%"
30,2,41.82,-87.18,73688,18422,3.70,7.7,2,309.0,"US 97%, CA 2%"
63,3,15.49,103.18,70588,17647,3.54,11.2,5,388.0,"TH 64%, VN 25%, KH 10%"
49,4,32.70,-86.06,69912,17478,3.51,14.7,1,346.0,US 100%
2,5,42.52,-76.53,65108,16277,3.27,18.0,3,302.0,"US 72%, CA 27%, BM 0%"
5,6,50.65,7.93,61104,15276,3.06,21.1,11,273.0,"DE 55%, FR 18%, NL 10%"
69,7,36.69,-80.86,60296,15074,3.02,24.1,1,273.0,US 100%
19,8,23.91,82.80,55380,13845,2.78,26.9,2,319.0,"IN 96%, NP 4%"
13,9,17.95,77.04,54768,13692,2.75,29.6,1,289.0,IN 100%


In [37]:
n = por_celda.imagenes.to_numpy()
x = np.sort(n)
gini = (2 * np.arange(1, K + 1) - K - 1) @ x / (K * x.sum())
p = n / n.sum()
celdas_efectivas = np.exp(-(p[p > 0] * np.log(p[p > 0])).sum())

resumen = pd.Series({
    "imágenes totales": n.sum(),
    "media por celda": n.mean(),
    "mediana por celda": np.median(n),
    "mínimo": n.min(),
    "máximo": n.max(),
    "máx / mín": n.max() / n.min(),
    "coef. de variación": n.std() / n.mean(),
    "Gini": gini,
    "celdas efectivas (exp entropía)": celdas_efectivas,
    "% imágenes en las 10 celdas más grandes": 100 * np.sort(p)[::-1][:10].sum(),
    "% imágenes en las 10 celdas más chicas": 100 * np.sort(p)[:10].sum(),
}).round(3)
display(resumen.to_frame("valor"))

,valor
imágenes totales,1993728.000
media por celda,28481.829
mediana por celda,25900.000
mínimo,132.000
máximo,79272.000
máx / mín,600.545
coef. de variación,0.723
Gini,0.407
celdas efectivas (exp entropía),53.163
% imágenes en las 10 celdas más grandes,32.328


### Gráficos

In [38]:
fig = go.Figure(go.Bar(
    x=ordenado.index.astype(str), y=ordenado.imagenes,
    marker_color="#1f77b4",
    customdata=np.column_stack([ordenado.panoramas, ordenado.pct_imagenes, ordenado.top_paises]),
    hovertemplate="celda %{x}<br>%{y:,} imágenes<br>%{customdata[0]:,} panoramas<br>%{customdata[1]}%<br>%{customdata[2]}<extra></extra>",
))
fig.add_hline(y=n.mean(), line_dash="dash", line_color="#555", annotation_text=f"media {n.mean():,.0f}")
fig.update_layout(title="Imágenes de train por cluster (ordenado)", xaxis_title="ind_cluster",
                  yaxis_title="imágenes", xaxis_type="category", height=450)
fig.show()

fig = go.Figure(go.Bar(
    x=por_celda.index.astype(str), y=por_celda.imagenes, marker_color="#1f77b4",
    hovertemplate="celda %{x}<br>%{y:,} imágenes<extra></extra>",
))
fig.update_layout(title="Imágenes de train por cluster (por índice)", xaxis_title="ind_cluster",
                  yaxis_title="imágenes", xaxis_type="category", height=400)
fig.show()

In [39]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Histograma de imágenes por celda", "Curva de Lorenz"))
fig.add_trace(go.Histogram(x=n, nbinsx=25, marker_color="#1f77b4"), row=1, col=1)
lorenz = np.concatenate([[0], np.cumsum(x) / x.sum()])
frac = np.linspace(0, 1, K + 1)
fig.add_trace(go.Scatter(x=frac, y=lorenz, mode="lines", line_color="#1f77b4", name="Lorenz"), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="#999"), name="igualdad"), row=1, col=2)
fig.update_xaxes(title_text="imágenes", row=1, col=1)
fig.update_yaxes(title_text="celdas", row=1, col=1)
fig.update_xaxes(title_text="fracción de celdas (de menor a mayor)", row=1, col=2)
fig.update_yaxes(title_text="fracción de imágenes", row=1, col=2)
fig.update_layout(height=420, showlegend=False, title=f"Desbalance entre celdas (Gini = {gini:.3f})")
fig.show()

### Mapa

Cada prototipo con tamaño proporcional a la raíz de la cantidad de imágenes y color en escala log.

In [40]:
fig = go.Figure(go.Scattergeo(
    lat=por_celda.lat, lon=por_celda.lon, mode="markers+text",
    text=por_celda.index.astype(str), textposition="top center", textfont=dict(size=8),
    marker=dict(size=6 + 40 * np.sqrt(n / n.max()), color=np.log10(n), colorscale="Viridis",
                colorbar=dict(title="imágenes", tickvals=np.log10([1e3, 3e3, 1e4, 3e4, 1e5]),
                              ticktext=["1k", "3k", "10k", "30k", "100k"]),
                line=dict(width=0.5, color="#333"), opacity=0.85),
    customdata=np.column_stack([n, por_celda.panoramas, por_celda.top_paises]),
    hovertemplate="celda %{text}<br>%{customdata[0]:,} imágenes<br>%{customdata[1]:,} panoramas<br>%{customdata[2]}<extra></extra>",
))
fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f1ede4", showocean=True,
                oceancolor="#dfeaf2", showcountries=True, countrycolor="#bbbbbb")
fig.update_layout(title=f"Imágenes de train por cluster, K={K}", height=600, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

## 4. Qué implica para un muestreo balanceado

Los clasificadores actuales usan ~20k imágenes balanceadas (~291 por celda). Con el train completo, un muestreo
**estrictamente balanceado sin repetir imágenes** queda limitado por la celda más chica. Para distintos
objetivos de imágenes por celda y época: cuántas celdas no alcanzan (tendrían que repetir imágenes),
cuántas imágenes únicas se usan y qué fracción del train completo representan.

In [41]:
objetivos = sorted({291, 1_000, 2_000, int(n.min()), 5_000, 10_000, int(np.median(n)), 20_000, int(n.mean())})
tabla_obj = pd.DataFrame([{
    "imagenes_por_celda": t,
    "total_epoca": t * K,
    "celdas_que_no_alcanzan": int((n < t).sum()),
    "imagenes_unicas_usadas": int(np.minimum(n, t).sum()),
    "pct_del_train": round(100 * np.minimum(n, t).sum() / n.sum(), 1),
    "repeticion_max_celda_chica": round(t / n.min(), 2),
} for t in objetivos]).set_index("imagenes_por_celda")
display(tabla_obj)
i_min, i_max = por_celda.imagenes.idxmin(), por_celda.imagenes.idxmax()
print(f"Celda más chica: {i_min} con {n.min():,} imágenes ({por_celda.loc[i_min, 'top_paises']})")
print(f"Celda más grande: {i_max} con {n.max():,} imágenes ({por_celda.loc[i_max, 'top_paises']})")

,total_epoca,celdas_que_no_alcanzan,imagenes_unicas_usadas,pct_del_train,repeticion_max_celda_chica
imagenes_por_celda,,,,,
132,9240,0,9240,0.5,1.00
291,20370,1,20211,1.0,2.20
1000,70000,2,68880,3.5,7.58
2000,140000,4,135744,6.8,15.15
5000,350000,8,329116,16.5,37.88
10000,700000,18,626868,31.4,75.76
20000,1400000,29,1081892,54.3,151.52
25900,1813000,35,1305440,65.5,196.21
28481,1993670,40,1390202,69.7,215.77


Celda más chica: 68 con 132 imágenes (RE 52%)
Celda más grande: 65 con 79,272 imágenes (FR 93%, ES 3%, IT 2%)


### Celdas más chicas y más grandes

In [42]:
cols = ["imagenes", "panoramas", "pct_imagenes", "paises", "dist_media_km", "top_paises"]
print("10 celdas con menos imágenes")
display(por_celda.nsmallest(10, "imagenes")[cols])
print("10 celdas con más imágenes")
display(por_celda.nlargest(10, "imagenes")[cols])

10 celdas con menos imágenes


,imagenes,panoramas,pct_imagenes,paises,dist_media_km,top_paises
ind_cluster,,,,,,
68,132,33,0.01,1,459.0,RE 52%
44,748,187,0.04,1,157.0,US 96%
42,1108,277,0.06,5,174.0,"PR 78%, DO 10%, CW 4%"
62,1756,439,0.09,1,447.0,AU 96%
67,2724,681,0.14,2,688.0,"SN 68%, ES 32%"
24,3208,802,0.16,2,585.0,"CA 48%, US 48%"
43,4476,1119,0.22,1,650.0,BR 99%
10,4964,1241,0.25,1,637.0,AU 100%
47,6552,1638,0.33,2,807.0,"RU 68%, MN 19%"


10 celdas con más imágenes


,imagenes,panoramas,pct_imagenes,paises,dist_media_km,top_paises
ind_cluster,,,,,,
65,79272,19818,3.98,6,276.0,"FR 93%, ES 3%, IT 2%"
30,73688,18422,3.70,2,309.0,"US 97%, CA 2%"
63,70588,17647,3.54,5,388.0,"TH 64%, VN 25%, KH 10%"
49,69912,17478,3.51,1,346.0,US 100%
2,65108,16277,3.27,3,302.0,"US 72%, CA 27%, BM 0%"
5,61104,15276,3.06,11,273.0,"DE 55%, FR 18%, NL 10%"
69,60296,15074,3.02,1,273.0,US 100%
19,55380,13845,2.78,2,319.0,"IN 96%, NP 4%"
13,54768,13692,2.75,1,289.0,IN 100%
